# 09.06 · Memory, state & human-in-the-loop

**Chapter:** Tool Use & Agents · **Milestone:** M4

## Goals
- [ ] Short-term memory (the conversation) vs long-term memory (facts that survive sessions)
- [ ] Build a long-term memory store: extract facts, embed, recall the relevant ones
- [ ] Persist and resume agent state with LangGraph checkpoints
- [ ] Require human approval before risky tool calls

## Resources
- [LangGraph – Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) · [Human-in-the-loop](https://docs.langchain.com/oss/python/langgraph/interrupts) — free
- [Mem0](https://github.com/mem0ai/mem0) · [Letta (MemGPT)](https://docs.letta.com/) · [Zep](https://www.getzep.com/) — free/OSS
- [Anthropic – Memory tool & context editing](https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/memory-tool) — free

| Memory | Lives in | Example |
|---|---|---|
| **Working / short-term** | the messages list (context window) | this conversation |
| **Long-term semantic** | a DB of facts/preferences, retrieved by similarity | "user is vegetarian" |
| **Episodic** | logs of past sessions/tasks | "last week we debugged the CSV export" |
| **State / checkpoints** | serialized agent state | resume a paused run after approval |

In [ ]:
import json
import sqlite3

import numpy as np
from pydantic import BaseModel

from aieng import DATA, rag
from aieng.llm import client

c, MODEL = client()
LOCAL = c.base_url.host == "localhost"
EXTRA = {"reasoning_effort": "none"} if LOCAL else {}

## 1. Long-term memory store
After each user message, an LLM extracts durable facts; they're stored with embeddings (SQLite here). Before answering, the most relevant
memories are recalled into the prompt. Same design as Mem0 / LangGraph Store, in ~40 lines.

In [ ]:
class Facts(BaseModel):
    facts: list[str]


class Memory:
    def __init__(self, path):
        self.db = sqlite3.connect(path)
        self.db.execute("CREATE TABLE IF NOT EXISTS memories (id INTEGER PRIMARY KEY, user TEXT, fact TEXT, embedding BLOB)")

    def remember(self, user, message):
        facts = c.chat.completions.parse(model=MODEL, temperature=0, max_tokens=400, response_format=Facts, **EXTRA, messages=[{"role": "user", "content":
            "Extract durable facts about the user worth remembering for future conversations (preferences, profile, goals). "
            f"Return an empty list if there are none.\n\nMessage: {message}"}]).choices[0].message.parsed.facts
        for f in facts:
            self.db.execute("INSERT INTO memories (user, fact, embedding) VALUES (?, ?, ?)", (user, f, rag.embed([f])[0].tobytes()))
        self.db.commit()
        return facts

    def recall(self, user, query, k=3):
        rows = self.db.execute("SELECT fact, embedding FROM memories WHERE user = ?", (user,)).fetchall()
        if not rows:
            return []
        E = np.stack([np.frombuffer(e, dtype=np.float32) for _, e in rows])
        scores = E @ rag.embed([query], query=True)[0]
        return [rows[i][0] for i in np.argsort(-scores)[:k]]


(DATA / "memory.sqlite").unlink(missing_ok=True)
mem = Memory(DATA / "memory.sqlite")
for m in ["Hi! I'm Ana, I live in Lima and I'm vegetarian.", "I'm learning AI engineering, mostly evenings after work.",
          "What's 2+2?", "I prefer short answers with code examples."]:
    print(f"{m!r} -> {mem.remember('ana', m)}")

A **new session** — the conversation history is gone, but memories come back:

In [ ]:
def chat_with_memory(user, message):
    memories = mem.recall(user, message)
    system = "You are a helpful assistant.\nWhat you know about the user:\n" + "\n".join(f"- {m}" for m in memories)
    reply = c.chat.completions.create(model=MODEL, max_tokens=300, **EXTRA, messages=[{"role": "system", "content": system},
                                                                                      {"role": "user", "content": message}]).choices[0].message.content
    return memories, reply


for q in ["Suggest a dinner recipe.", "How should I schedule my study time?"]:
    memories, reply = chat_with_memory("ana", q)
    print(f"Q: {q}\n  recalled: {memories}\n  A: {reply[:200]!r}\n")

Real memory systems also handle **updates and contradictions** ("I moved to Cusco" should replace "lives in Lima"), deduplication, expiry,
and user control (view/delete memories — also a privacy requirement).

## 2. Checkpoints: persist and resume agent state (LangGraph)
A checkpointer saves the graph state after every step, keyed by `thread_id`. Restart the process, pass the same `thread_id`, continue.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver  # swap for SqliteSaver / PostgresSaver to survive restarts

llm = ChatOpenAI(model=MODEL, base_url=str(c.base_url), api_key=c.api_key, temperature=0, **EXTRA)
checkpointer = InMemorySaver()
agent = create_agent(llm, tools=[], checkpointer=checkpointer, system_prompt="Be brief.")

cfg = {"configurable": {"thread_id": "ana-1"}}
agent.invoke({"messages": [{"role": "user", "content": "My favourite city is Arequipa."}]}, cfg)
out = agent.invoke({"messages": [{"role": "user", "content": "What's my favourite city?"}]}, cfg)
print("same thread :", out["messages"][-1].content)
out = agent.invoke({"messages": [{"role": "user", "content": "What's my favourite city?"}]}, {"configurable": {"thread_id": "other"}})
print("other thread:", out["messages"][-1].content)
print("checkpoints saved for ana-1:", len(list(checkpointer.list(cfg))))

## 3. Human-in-the-loop approval
Classify tools by risk. Read-only tools run freely; side-effecting tools (send email, pay, delete, deploy) pause for approval.
In a notebook we pass an `approve` callback; in an app it's a UI button, a Slack message, or LangGraph's `interrupt()` (which checkpoints
the run and resumes later with `Command(resume=...)`).

In [ ]:
def send_email(to: str, subject: str, body: str) -> str:
    return f"(pretend) email sent to {to}"


def lookup_order(order_id: str) -> str:
    return json.dumps({"order_id": order_id, "status": "delivered", "customer_email": "ana@example.com"})


TOOLS = {"send_email": (send_email, "risky"), "lookup_order": (lookup_order, "safe")}
SCHEMAS = [
    {"type": "function", "function": {"name": "lookup_order", "description": "Look up an order by id.",
                                      "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "send_email", "description": "Send an email to a customer.",
                                      "parameters": {"type": "object", "properties": {"to": {"type": "string"}, "subject": {"type": "string"},
                                                     "body": {"type": "string"}}, "required": ["to", "subject", "body"]}}},
]


def run_with_approval(task, approve, max_steps=6):
    messages = [{"role": "user", "content": task}]
    for _ in range(max_steps):
        msg = c.chat.completions.create(model=MODEL, messages=messages, tools=SCHEMAS, temperature=0, max_tokens=600, **EXTRA).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            fn, risk = TOOLS[call.function.name]
            args = json.loads(call.function.arguments)
            if risk == "risky" and not approve(call.function.name, args):
                result = "DENIED by human reviewer. Do not retry; tell the user what you would have done."
            else:
                result = fn(**args)
            print(f"  {call.function.name}({args}) [{risk}] -> {result[:80]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "step limit"


def deny_all(name, args):
    print(f"  ⏸  approval requested: {name} {args} -> denied")
    return False


print(run_with_approval("Look up order 991 and email the customer that it was delivered.", approve=deny_all))

## Exercises
1. Handle contradictions in `Memory`: before inserting a fact, recall similar ones (similarity > 0.8) and ask the LLM whether the new fact
   replaces, updates or adds. Test with "I moved to Cusco".
2. Replace `InMemorySaver` with `SqliteSaver` (`uv add langgraph-checkpoint-sqlite`), restart the kernel, and resume the thread.
3. Implement approval with LangGraph's `interrupt()` inside a tool, then resume with `Command(resume={"approved": True})`.

In [ ]:
# your code here